<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/agent_registration_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting up Okta for A2A Cross App Access

This notebook provides an API-based companion for configuring the [A2A Colab](https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/cross_app_a2a.ipynb).
This notebook is the API-based companion to the [developer guide](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm) for adding an AI Agent manually. 

## Prerequisites
1. An Okta Preview environment with O4AA enabled. 
2. A general understanding of how to make API calls and use this notebook.

> IMPORTANT
> *This guide utilizes an API token rather than OAuth-based public/private key.*
>
> In a production environment, we strongly encourage the use of `Public key / Private key` authentication method.
> <br>

## Complete Registration Flow (10 Steps)

1. **Create an API Key**
2. **Create a Client Application**
3. **Create a Custom Authorization Server**
   1. Create a custom claim
   2. Create an access policy & rule
4. **Create a 'HR Benefits MCP' Resource**
   1. Create a Custom Authorization Server
   2. Create a custom claim
   3. Create an access policy & rule
5. **Register Agent J**
   1. Assign an owner
   2. Configure client registration
   3. Configure Agent Caller
   4. Activate Agent?
6. **Register Agent K**
   1. Assign an owner
   2. Configure client registration
   3. Configure Agent Caller
   4. Add 'HR Benefits MCP' as a resource connection
   5. Activate Agent?

### Important Training Notes
- This notebook intentionally uses direct HTTP calls instead of an SDK to teach protocol-level behavior.
- This notebook utilizes an API token *for simplicity*. 
  - **In production, we strongly recommend using a public/private key.**
- This notebook creates agents that use a client secret. 
  - **In production, we strongly recommend using a public/private key.**

<br>

---

## STEP 1: Create an API Token

1. Follow the Okta guide for [creating an API token](https://help.okta.com/oie/en-us/content/topics/security/api.htm#security-api__create-okta-api-token) 
2. Paste the API token below. 
3. Enter your Okta domain. 
4. Click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to set the configuration.

In [ ]:
# @title { display-mode: "form" }

from pathlib import Path
import importlib
import sys
import requests
from typing import Any, TypedDict, Literal
from urllib.parse import urlparse
from typing import cast

OKTA_DOMAIN = "https://atko-ai.oktapreview.com"  # @param { type: "string", placeholder: "Enter your entire Okta domain (including https)"}
API_TOKEN = "00ND3-l8KMTjtOYIVpWPfWMb_TRjK6U3gN4D_fDpF3"  # @param {type: "string", placeholder: "Enter your Okta API token"}

# @markdown _Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!_

# =============== LOAD NOTEBOOK UTILITY FUNCTIONS ==============
# In hosted notebooks, fetch utils.py before importing it. Locally, prefer the checked-out copy.
UTILS_URL = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/feature%2Fa2a/utils.py"
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "utils.py").exists()), None)
IS_LOCAL = repo_root is not None

if repo_root is None:
    import requests

    response = requests.get(UTILS_URL, timeout=15)
    response.raise_for_status()
    Path("utils.py").write_text(response.text)
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import utils

importlib.reload(utils)

print(f"✅ Utilities loaded from {utils.__file__}")
# ==========================

try:
    utils.validate_config(locals(), "a2a_setup")
except ValueError as e:
    print(e)
    raise SystemExit("❌ Configuration validation failed. Please fix the above errors and re-run the cell.")


class State(TypedDict):
    api_token: str
    okta_domain: str
    org_id: str
    client_app: utils.types.OidcAppState
    agent_j: utils.types.AIAgentState
    agent_k: utils.types.AIAgentState
    resource: utils.types.AuthorizationServer | None
    output: dict[str, Any]


previous_state = globals().get("STATE", {})

reset_state: bool | None = None

if len(previous_state.keys()) > 0:
    reset_state = previous_state.get("okta_domain") is not None and previous_state.get("okta_domain") != OKTA_DOMAIN

    if reset_state:
        print("🔄 Resetting state due to change in OKTA_DOMAIN.")
        print("❌ You will need to rerun all steps!")
    else:
        print("ℹ️ Retaining previous state.")

DEFAULT_OUTPUT = {
    "OKTA_DOMAIN": OKTA_DOMAIN,
    "DOMAIN_A_ISSUER": "",
    "REDIRECT_URI": "",
    "CLIENT_ID": "",
    "CLIENT_SECRET": "",
    "PRINCIPAL_A_ID": "",
    "PRINCIPAL_A_RESOURCE_URI": "",
    "PRINCIPAL_A_SECRET": "",
    "DOMAIN_B_ISSUER": "",
    "PRINCIPAL_B_ID": "",
    "PRINCIPAL_B_RESOURCE_URI": "",
    "PRINCIPAL_B_SECRET": "",
    "RESOURCE_ISSUER": "",
    "RESOURCE_SERVER_AUDIENCE": "",
}

STATE: State = {
    "api_token": API_TOKEN,
    "okta_domain": OKTA_DOMAIN,
    "org_id": "" if reset_state else previous_state.get("org_id", ""),
    "client_app": {} if reset_state else previous_state.get("client_app", {}),
    "agent_j": {} if reset_state else previous_state.get("agent_j", {}),
    "agent_k": {} if reset_state else previous_state.get("agent_k", {}),
    "resource": None if reset_state else previous_state.get("resource", None),
    "output": (DEFAULT_OUTPUT if reset_state else previous_state.get("output", DEFAULT_OUTPUT)),
}


# ========== HELPER FUNCTIONS ==========
def okta_request(
    path: str,
    method: str = "POST",
    *,
    okta_domain: str | None = OKTA_DOMAIN,
    json: Any = None,
    params: dict[str, Any] | None = None,
    timeout: float = 30,
) -> requests.Response:
    """Call the Okta management API using this notebook's API_TOKEN.

    Example: apps = okta_request("GET", "/api/v1/apps", params={"limit": 20}).json()
    For POST or PUT, pass the payload as json=payload.
    HTTP failures raise requests.HTTPError; successful calls return a Response.
    A 204 response has no JSON body. Inspect response.links for pagination;
    this helper returns one page and does not automatically retry mutations.
    API_TOKEN uses SSWS authentication, not an OAuth Bearer access token.
    """
    return utils.okta_request(
        method,
        path,
        okta_domain=okta_domain,
        api_token=API_TOKEN,
        json=({key: value for key, value in json.items() if value is not None} if isinstance(json, dict) else json),
        params=params,
        timeout=timeout,
    )


def get_okta_partition(okta_domain: str | None = None) -> str:
    if okta_domain is None:
        okta_domain = STATE.get("okta_domain", OKTA_DOMAIN)

    """Infer ORN partition from Okta domain."""
    hostname = urlparse(okta_domain).hostname or ""
    if hostname.endswith(".oktapreview.com"):
        return "oktapreview"
    if hostname.endswith(".okta-emea.com"):
        return "okta-emea"
    if hostname.endswith(".okta.com"):
        return "okta"
    raise ValueError(f"Unsupported Okta domain for ORN partition derivation: {okta_domain}")


def get_org_id():
    if not STATE.get("org_id"):
        response = okta_request(path=f"/.well-known/okta-organization", method="GET")
        org_id = response.json().get("id")
        STATE["org_id"] = org_id
    return STATE["org_id"]


def get_orn(
    service: Literal["directory", "idp", "workflow", "governance"],
    object_type: Literal[
        "apps",
        "workload-principals",
        "groups",
        "users",
        "resource-servers",
        "authorization_servers",
        "identity_provider",
    ],
    *,
    partition: str | None = None,
    org_id: str | None = None,
    app_name: str | None = None,
    object_id: str | None = None,
) -> str:
    orn = f"orn:{partition or get_okta_partition()}:{service}:{org_id or get_org_id()}:{object_type}"

    if app_name:
        orn += f":{app_name}"

    if object_id:
        orn += f":{object_id}"
    return orn

## Step 2: Create a Client Application

*Learn more about creating a custom OIDC application via the UI [here](https://help.okta.com/oie/en-us/content/topics/apps/apps_app_integration_wizard_oidc.htm).*

The following code will create a custom OIDC application in Okta. Click `Show Code` to see the details of the API request or to modify the parameters (at your own risk).

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

CLIENT_APP_NAME = "Chat App"  # @param { type: "string", placeholder: "Enter a name for your client/web application." }
# @markdown <sup><code>label</code></sup>
# @markdown <br>
# @markdown <sub><em><b>NOTE:</b> This notebook currently only supports `client_secret_basic` authentication at this time.</em></sub>


def create_app(name: str) -> utils.types.OidcApp:
    print(f"⏳ Creating client application '{name}'...")

    result = okta_request(
        path="/api/v1/apps",
        json={
            "label": name,
            "signOnMode": "OPENID_CONNECT",
            "credentials": {"oauthClient": {"token_endpoint_auth_method": "client_secret_basic"}},
            "name": "oidc_client",
            "settings": {
                "oauthClient": {
                    "application_type": "web",
                    "grant_types": ["authorization_code", "refresh_token"],
                    "redirect_uris": ["http://localhost:8080/authorization-code/callback"],
                    "response_types": ["code"],
                }
            },
        },
    )

    data = cast(utils.types.OidcApp, result.json())

    print(f"✅ Client application created: {data.get("label")}")

    return data


client_app = STATE.get("client_app", {}).get("app", {})
client_id = client_app.get("id", None)
client_secret = client_app.get("credentials", {}).get("oauthClient", {}).get("client_secret", None)

if not client_app:
    client_app = create_app(CLIENT_APP_NAME)

    client_id = client_app.get("id", None)
    client_secret = client_app.get("credentials", {}).get("oauthClient", {}).get("client_secret", None)

    STATE["output"]["CLIENT_ID"] = client_id
    STATE["output"]["CLIENT_SECRET"] = client_secret
    STATE["output"]["REDIRECT_URI"] = client_app.get("settings").get("oauthClient").get("redirect_uris")[0]

elif client_app.get("id") and client_app.get("label") == CLIENT_APP_NAME:
    print(f"⚠️ Client application '{CLIENT_APP_NAME}' already exists!")
    print("Change the CLIENT_APP_NAME value if you would like to create a new application.")

if client_app.get("orn") is None:
    print(f"⚠️ Client application '{CLIENT_APP_NAME}' does not have an ORN!")
    print("⏳ Generating ORN...")

    client_app["orn"] = get_orn(
        service="idp",
        object_type="apps",
        app_name="oidc_client",
        object_id=client_app.get("id"),
    )

STATE["client_app"]["app"] = cast(utils.types.OidcApp, client_app)

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → CLIENT_ID: {client_id}")
print(f"    → CLIENT_SECRET: {client_secret}")

## Step 3: Create a Custom Authorization Server

TODO


### 1. Create a custom authorization Server

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-auth-server.

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

CLIENT_AUTH_SERVER_NAME = "Chat App"  # @param { type: "string", placeholder: "Enter a name for your client/web application\'s authorization server." }
# @markdown <sup><code>name</code></sup>


def create_auth_server(name: str, description: str, audiences: list[str]) -> utils.types.AuthorizationServer:
    try:
        client_authz_result = okta_request(
            path="/api/v1/authorizationServers",
            json={
                "name": name,
                "description": description,
                "audiences": audiences,
            },
        )

        auth_server_response = client_authz_result.json()

        if not isinstance(auth_server_response, dict):
            raise ValueError("Authorization Server response was not a JSON object.")

        resource_server = cast(utils.types.AuthorizationServer, auth_server_response)

        print(f"✅ Custom Authorization Server Created: {resource_server.get('name')}")

        # Generate ORN. We will need it later.
        if not resource_server.get("orn"):
            resource_server.update(
                {
                    "orn": get_orn(
                        service="idp",
                        object_type="authorization_servers",
                        object_id=resource_server.get("id"),
                    )
                }
            )

        return resource_server

    except Exception as e:
        print(f"❌ Error creating authorization server: {e}")
        print("💡 Try changing the name of the authorization server and running again.")
        raise


client_app_resource_server = create_auth_server(
    name=CLIENT_AUTH_SERVER_NAME,
    description=f"{CLIENT_AUTH_SERVER_NAME} Authorization Server",
    audiences=["api://default"],
)

STATE["client_app"]["resource_server"] = client_app_resource_server
STATE["output"]["DOMAIN_A_ISSUER"] = client_app_resource_server.get("issuer")


print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → DOMAIN_A_ISSUER: {client_app_resource_server.get('issuer')}")

### 2. Create a custom scope

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-scopes.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

# @markdown <sub><em>Separate multiple scope names with commas.</em></sub>
CLIENT_AUTH_SERVER_SCOPES = "agent:invoke"  # @param { type: "string", placeholder: "Enter scope names for your Client\'s required scopes." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
# @markdown <br><sub><em>Separate multiple scope descriptions with commas in the same order as the name(s)</em></sub>
CLIENT_AUTH_SERVER_SCOPE_DESCRIPTIONS = "Invoke AI Agent"  # @param { type: "string", placeholder: "Enter a description for your Client\'s required scope." }
# @markdown <sup><code>description</code></sup>


def create_scopes(
    auth_server: utils.types.AuthorizationServer,
    scopes: str,
    descriptions: str,
    displayNames: str | None = None,
) -> str:

    scope_list = [s.strip() for s in scopes.split(",")]
    description_list = [d.strip() for d in descriptions.split(",")]
    displayName_list = [dn.strip() for dn in displayNames.split(",")] if displayNames else scope_list

    try:
        scopes_output = []

        for scope_index, name in enumerate(scope_list):
            description = description_list[scope_index] or name if scope_index < len(description_list) else name
            displayName = displayName_list[scope_index] or name if scope_index < len(displayName_list) else name
            result = okta_request(
                path=f"/api/v1/authorizationServers/{auth_server.get('id')}/scopes",
                json={
                    "name": name,
                    "description": description,
                    "consent": "REQUIRED",
                    "displayName": displayName or name,
                },
            )
            scope_result = cast(dict[str, Any], result.json())
            scope_name = cast(str, scope_result.get("name"))
            print(f"✅ Custom Scope Created: {scope_name}")

            scopes_output.append(scope_name)

        return ", ".join(scopes_output)

    except Exception as e:
        print(f"❌ Error creating custom scopes: {e}")
        raise


if (client_app_resource_server := STATE["client_app"].get("resource_server")) is None:
    raise ValueError("❌ Client application resource server must be created before setting up its scopes. " "Rerun the client authorization server setup cell.")

client_app_scopes = create_scopes(
    auth_server=client_app_resource_server,
    scopes=CLIENT_AUTH_SERVER_SCOPES,
    descriptions=CLIENT_AUTH_SERVER_SCOPE_DESCRIPTIONS,
    displayNames=CLIENT_AUTH_SERVER_SCOPES,
)

STATE["output"]["CLIENT_SCOPES"] = client_app_scopes

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → CLIENT_SCOPES: {STATE['output'].get('CLIENT_SCOPES')}")

### 3. Create an access policy & rule

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-access-policies.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

CLIENT_AUTH_SERVER_POLICY_NAME = "Client App Policy"  # @param { type: "string", placeholder: "Enter a name for your Client\'s access policy." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
CLIENT_AUTH_SERVER_POLICY_DESCRIPTION = "Policy for Client App authorization server"  # @param { type: "string", placeholder: "Enter a description for your Client\'s access policy rule." }
CLIENT_AUTH_SERVER_RULE_NAME = "Invoke Agent"  # @param { type: "string", placeholder: "Enter a name for your Client\'s access policy rule." }
# @markdown <sup><code>description</code></sup>


# ========== HELPER FUNCTION ==========
def create_policy(
    auth_server: utils.types.AuthorizationServer,
    name: str,
    description: str,
    rule_name: str,
    scopes: list[str],
) -> str:
    auth_server_id = auth_server.get("id")

    if not auth_server_id:
        raise ValueError("Authorization server is missing its ID.")

    access_policy_id = auth_server.get("resource_access_policy_id")

    if not access_policy_id:
        policy_result = okta_request(
            path=f"/api/v1/authorizationServers/{auth_server_id}/policies",
            json={
                "type": "OAUTH_AUTHORIZATION_POLICY",
                "name": name,
                "description": description,
                "conditions": {"clients": {"include": ["ALL_CLIENTS"]}},
                "status": "ACTIVE",
            },
        )
        policy_body = policy_result.json()
        access_policy_id = policy_body.get("id")
        if not isinstance(access_policy_id, str) or not access_policy_id:
            raise ValueError("Created access policy response did not include an ID.")

        auth_server["resource_access_policy_id"] = access_policy_id
        print(f"✅ Access Policy Created: {policy_body.get('name')}")
    else:
        print(f"ℹ️ Access policy already created: {access_policy_id}")

    rule_result = okta_request(
        path=f"/api/v1/authorizationServers/{auth_server_id}/policies/{access_policy_id}/rules",
        json={
            "type": "RESOURCE_ACCESS",
            "name": rule_name,
            "conditions": {
                "people": {
                    "users": {"include": [], "exclude": []},
                    "groups": {"include": ["EVERYONE"], "exclude": []},
                },
                "grantTypes": {
                    "include": [
                        "urn:ietf:params:oauth:grant-type:token-exchange",
                        "urn:ietf:params:oauth:grant-type:jwt-bearer",
                    ],
                },
                "scopes": {"include": scopes},
            },
            "status": "ACTIVE",
        },
    )

    print(f"✅ Access Policy Rule Created: {rule_result.json().get('name')}")
    return access_policy_id


# ========== END HELPER FUNCTION ==========

if (client_app_resource_server := STATE["client_app"].get("resource_server")) is None:
    raise ValueError("Client app resource server not found. Create it first.")

client_app_resource_server_policy_id = create_policy(
    auth_server=client_app_resource_server,
    name=CLIENT_AUTH_SERVER_POLICY_NAME,
    description=CLIENT_AUTH_SERVER_POLICY_DESCRIPTION,
    rule_name=CLIENT_AUTH_SERVER_RULE_NAME,
    scopes=[scope.strip() for scope in STATE["output"].get("CLIENT_SCOPES", "[*]").strip().strip("[]").split(",") if scope.strip()],
)

client_app_resource_server["resource_access_policy_id"] = client_app_resource_server_policy_id

STATE["client_app"]["resource_server"] = client_app_resource_server

## Step 4: Create an MCP Resource

TODO

### 1. Create a custom authorization Server

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-auth-server.

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

MCP_RESOURCE_SERVER_NAME = "Benefits MCP Server2"  # @param { type: "string", placeholder: "Enter a name for your MCP resource\'s authorization server." }
# @markdown <sup><code>name</code></sup>
MCP_RESOURCE_SERVER_AUDIENCE = "https://benefits.streamward.com"  # @param { type: "string", placeholder: "Enter an audience value for your MCP resource." }
# @markdown <sup><code>audiences</code></sup>

mcp_resource_server = create_auth_server(
    name=MCP_RESOURCE_SERVER_NAME,
    description=f"{MCP_RESOURCE_SERVER_NAME} Authorization Server",
    audiences=[MCP_RESOURCE_SERVER_AUDIENCE],
)

STATE["resource"] = mcp_resource_server
STATE["output"]["RESOURCE_ISSUER"] = mcp_resource_server.get("issuer")
STATE["output"]["RESOURCE_SERVER_AUDIENCE"] = mcp_resource_server.get("audiences", [MCP_RESOURCE_SERVER_AUDIENCE])[0]

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → RESOURCE_ISSUER: {mcp_resource_server.get('issuer')}")
print(f"    → RESOURCE_SERVER_AUDIENCE: {mcp_resource_server.get('audiences', [MCP_RESOURCE_SERVER_AUDIENCE])[0]}")

### 2. Create a custom scope

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-scopes.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

# @markdown <sub><em>Separate multiple scope names with commas.</em></sub>
MCP_RESOURCE_SERVER_SCOPES = "mcp:read"  # @param { type: "string", placeholder: "Enter scope names for your MCP resource\'s required scopes." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
# @markdown <br><sub><em>Separate multiple scope descriptions with commas in the same order as the name(s)</em></sub>
MCP_RESOURCE_SERVER_SCOPE_DESCRIPTIONS = "Read-only access"  # @param { type: "string", placeholder: "Enter descriptions for your MCP resource\'s required scopes." }
# @markdown <sup><code>description</code></sup>

if (mcp_resource_server := STATE.get("resource")) is None:
    raise ValueError("❌ MCP resource server must be created before setting up its scopes. " "Rerun the MCP authorization server setup cell.")

mcp_resource_server_scopes = create_scopes(
    auth_server=mcp_resource_server,
    scopes=MCP_RESOURCE_SERVER_SCOPES,
    descriptions=MCP_RESOURCE_SERVER_SCOPE_DESCRIPTIONS,
    displayNames=MCP_RESOURCE_SERVER_SCOPES,
)

STATE["output"]["PRINCIPAL_B_SCOPES"] = mcp_resource_server_scopes

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_B_SCOPES: {STATE['output'].get('PRINCIPAL_B_SCOPES')}")

### 3. Create an access policy & rule

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-access-policies.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

MCP_RESOURCE_SERVER_POLICY_NAME = "Benefits MCP Policy"  # @param { type: "string", placeholder: "Enter a name for your MCP resource\'s access policy." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
MCP_RESOURCE_SERVER_POLICY_DESCRIPTION = "Policy for Benefits MCP server"  # @param { type: "string", placeholder: "Enter a description for your MCP resource\'s access policy rule." }
MCP_RESOURCE_SERVER_RULE_NAME = "Read-only access"  # @param { type: "string", placeholder: "Enter a name for your MCP resource\'s access policy rule." }
# @markdown <sup><code>description</code></sup>

if (mcp_resource_server := STATE["resource"]) is None:
    raise ValueError("MCP resource server not found in STATE. Make sure you have created the MCP resource server before setting up the policy.")

mcp_resource_server_policy_id = create_policy(
    auth_server=mcp_resource_server,
    name=MCP_RESOURCE_SERVER_POLICY_NAME,
    description=MCP_RESOURCE_SERVER_POLICY_DESCRIPTION,
    rule_name=MCP_RESOURCE_SERVER_RULE_NAME,
    scopes=[scope.strip() for scope in STATE["output"].get("PRINCIPAL_B_SCOPES", "[*]").strip().strip("[]").split(",") if scope.strip()],
)

mcp_resource_server["resource_access_policy_id"] = mcp_resource_server_policy_id

STATE["resource"] = mcp_resource_server

## Step 5: Register Agent J

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

import time

PRINCIPAL_A_NAME = "Agent J"  # @param { type: "string", placeholder: "Enter a name for Principal A." }
# @markdown <sup><code>profile.name</code></sup>
PRINCIPAL_A_DESCRIPTION = "Special Agent J"  # @param { type: "string", placeholder: "Enter a description for Principal A." }
# @markdown <sup><code>profile.description</code></sup>
PRINCIPAL_A_RESOURCE_URI = "https://agent_j"  # @param { type: "string", placeholder: "Enter the resource URI for Principal A." }


# ========== HELPER FUNCTIONS ==========
def register_agent(
    agent: utils.types.AIAgent,
    *,
    name: str | None = None,
    description: str | None = None,
    resource_uri: str | None = None,
) -> utils.types.AIAgent:
    """Reuse, resume, or register in place; None overrides retain saved values."""

    current_name = profile.get("name") if (profile := agent.get("profile")) else None
    current_description = profile.get("description") if profile else current_name
    current_resource_uri = agent.get("resource_uri")

    if has_changed := (agent.get("id") and ((name and name != current_name) or (description and description != current_description) or (resource_uri and resource_uri != current_resource_uri))):
        print("⚠️ Notebook input values changed! Clearing existing agent data & registering new agent.")

    operation_url = None if has_changed else (current_operation.get("url") if ((current_operation := agent.get("current_operation")) and current_operation.get("status") in ["IN_PROGRESS", "SCHEDULED"]) else None)

    if has_changed or not agent.get("id"):

        agent = {}

        new_name = name or current_name
        new_description = description or current_description or new_name

        result = okta_request(
            path="/workload-principals/api/v1/ai-agents",
            json={
                "profile": {
                    "name": new_name,
                    "description": new_description,
                },
                "resourceUrl": resource_uri,
            },
        )

        if not isinstance(operation_url := result.headers.get("Location"), str) or not operation_url.strip():
            raise ValueError("Create AI Agent response did not include a Location header.")

        print("⏳ AI Agent registration started. Waiting for completion...")

        agent["current_operation"] = cast(
            utils.types.AIAgentOperation,
            {"status": "IN_PROGRESS", "url": operation_url},
        )

    if operation_url:
        operation = cast(utils.types.AIAgentOperation, complete_operation(operation_url))

        print(f"✅ AI Agent registration completed!")

        agent_url = resource_self.get("href") if (resource_self := (resource_links.get("self") if (resource_links := operation.get("resource").get("_links")) else None)) else None

        if not agent_url:
            raise ValueError("Completed operation did not include an agent self link.")

        print(f"⌛️ Fetching agent details...")

        response = okta_request(path=agent_url, method="GET", okta_domain=None)

        agent.update(cast(utils.types.AIAgent, response.json()))
        agent["resource_uri"] = resource_uri if resource_uri else current_resource_uri if current_resource_uri else ""

    else:
        print("⚠️ Agent already exists. Skipping agent registration.")

    agent.pop("current_operation", None)

    # Generate ORN if not present. We will need it later
    if not agent.get("orn"):
        print("⏳ Generating missing ORN for the agent...")
        agent.update(
            {
                "orn": get_orn(
                    service="directory",
                    object_type="workload-principals",
                    app_name="ai-agents",
                    object_id=agent.get("id"),
                )
            }
        )

    return agent


def complete_operation(
    operation_url: str | None,
    max_poll_attempts: int = 60,
    poll_inverval_seconds: int = 2,
) -> utils.types.Operation:
    if not operation_url:
        raise ValueError("An operation_url must be provided.")

    try:
        print(f"Polling operation URL: {operation_url} (max attempts: {max_poll_attempts}, interval: {poll_inverval_seconds}s)")

        for attempt in range(1, max_poll_attempts + 1):
            operation_response = okta_request(path=operation_url, method="GET", okta_domain=None)

            operation = cast(utils.types.Operation, operation_response.json())

            status = operation.get("status")

            print(f"Operation status: {status} ({attempt}/{max_poll_attempts})")

            if status == "COMPLETED":
                return operation

            if status in {"FAILED", "CANCELED"}:
                raise RuntimeError(f"Operation ended with status {status}: {operation}")

            if attempt < max_poll_attempts:
                time.sleep(poll_inverval_seconds)

        raise TimeoutError(f"Operation did not complete after " f"{max_poll_attempts * poll_inverval_seconds} seconds. " "Rerun this cell to continue polling.")
    except Exception as e:
        print(f"❌ Error completing operation:\n   {e}")
        raise


# ========== END HELPER FUNCTIONS ==========

agent_j = STATE.get("agent_j")
if agent_j is None:
    agent_j = cast(utils.types.AIAgentState, {})
    STATE["agent_j"] = agent_j

try:
    agent_j = cast(
        utils.types.AIAgentState,
        register_agent(
            agent_j,
            name=PRINCIPAL_A_NAME,
            description=PRINCIPAL_A_DESCRIPTION,
            resource_uri=PRINCIPAL_A_RESOURCE_URI,
        ),
    )

    STATE["agent_j"] = agent_j
finally:
    if agent_j.get("id") != STATE["output"].get("PRINCIPAL_A_ID"):
        STATE["output"]["PRINCIPAL_A_SECRET"] = ""

    STATE["output"]["PRINCIPAL_A_ID"] = agent_j.get("id", "")
    STATE["output"]["PRINCIPAL_A_RESOURCE_URI"] = agent_j.get("resource_uri", "")

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → PRINCIPAL_A_ID: {agent_j.get('id')}")
print(f"    → PRINCIPAL_A_RESOURCE_URI: {agent_j.get('resource_uri')}")

### Assign an owner (optional)

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Assign%20owners%20(optional)

### Configure client registration

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Add%20client%20registration%20details

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown <br><br>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.
# @markdown <br><br>
# @markdown <em>Only <code>client_secret_basic</code> is supported at this time.</em>
# @markdown <br><br>
# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

PRINCIPAL_A_AUTH_METHOD = "client_secret_basic"  # @param ["client_secret_basic"]
# @markdown <sup><code>client.token_endpoint_auth_method</code></sup>


def create_credentials(id: str) -> str:
    try:

        print(f"⏳ Creating credentials for principal {id}...")

        okta_request(
            path=f"/workload-principals/api/v1/ai-agents/{id}/credentials/secrets",
            method="POST",
            json={},
        )

        response = okta_request(
            path=f"/workload-principals/api/v1/ai-agents/{id}/credentials/secrets",
            method="GET",
        )

        credentials = response.json()

        if len(credentials) > 1:
            print(f"   ⚠️ More than one credential found for principal {id}. Returning most recently created.")

        credential_result = credentials[-1] if credentials else None

        if credential_result is None:
            raise ValueError(f"No credentials found for principal {id}. Make sure you have created an agent and entered the correct information before re-running this cell!")

        # Activate the client secret
        client_secret = credential_result.get("client_secret", "")

        if not client_secret:
            raise ValueError(f"Client secret not found for principal {id}.")

        print(f"✅ Client secret created for principal {id}!")
        # Get the client so we can update it.
        client = okta_request(path=f"/oauth2/v1/clients/{id}", method="GET")

        client_data = client.json()
        client_data.update(
            {
                "client_secret": client_secret,
                "client_secret_expires_at": 0,
                "token_endpoint_auth_method": "client_secret_basic",
            }
        )

        print(f"\n⏳ Activating the client secret...")

        # Update the client. This activates the credential.
        okta_request(path=f"/oauth2/v1/clients/{id}", method="PUT", json=client_data)

        print(f"✅ Client secret activated!")

        return client_secret

    except Exception as e:
        print(f"Error creating secret for principal {id}: {e}")
        return ""


assert isinstance(principal_a_id := STATE["output"].get("PRINCIPAL_A_ID"), str), "PRINCIPAL_A_ID is missing!"

principal_a_secret = create_credentials(principal_a_id)
STATE["output"]["PRINCIPAL_A_SECRET"] = principal_a_secret

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_A_SECRET: {principal_a_secret}")

### Activate Agent J

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Activate%20an%20AI%20agent



In [ ]:
# ========= HELPER FUNCTION ==========
def activate_agent(id: str):

    print(f"⏳ Activating agent: {id}")

    response = okta_request(
        path=f"/workload-principals/api/v1/ai-agents/{id}/lifecycle/activate",
        method="POST",
    )

    if response.status_code == 202:
        print("✅ Agent activated successfully.")
    else:
        print(f"❌Failed to activate agent. Status code: {response.status_code}")


# ========== END HELPER FUNCTION ==========

assert isinstance(principal_a_id := STATE["output"].get("PRINCIPAL_A_ID"), str), "PRINCIPAL_A_ID is missing!"

activate_agent(principal_a_id)

### Configure Agent Caller

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Configure%20machine%20access

In [ ]:
# ========== HELPER FUNCTIONS ==========
def create_a2a_resource(wlp_id: str, auth_server_orn: str):

    print(f"⌛️ Creating A2A resource with ORN: {auth_server_orn}")

    response = okta_request(
        path=f"/resource-servers/api/v1/a2a-servers/{wlp_id}/authorization-servers",
        method="POST",
        json={"orn": auth_server_orn, "type": "OKTA"},
    )

    if response.status_code not in [202, 204]:
        raise ValueError(f"❌ Failed to create A2A resource. Status code: {response.status_code}, Response: {response.text}")

    print(f"⌛️ Waiting for operation to complete...")

    complete_operation(response.headers.get("Location"))

    print(f"✅ Resource created successfully: {auth_server_orn}")


def create_delegation_link(
    client_orn: str,
    resource_orn: str,
    *,
    auth_server_orn: str | None = None,
    token_type: Literal["ID_TOKEN", "ACCESS_TOKEN"] = "ACCESS_TOKEN",
):
    print(f"⌛️ Creating delegation link from client {client_orn} to resource {resource_orn}{f' with auth server {auth_server_orn}' if auth_server_orn else ''}")

    okta_request(
        path="/workload-principals/api/v1/delegation-links",
        method="POST",
        json={
            "from": {
                "type": "OKTA_AUTHORIZATION_SERVER",
                "clientOrn": client_orn,
                "tokenType": token_type,
            },
            "to": {
                "resourceOrn": resource_orn,
                "authorizationServerOrn": auth_server_orn,
            },
        },
    )

    print(f"✅ Delegation link created successfully!")


def create_connection(
    wlp_id: str,
    connection_type: Literal[
        "IDENTITY_ASSERTION_A2A_SERVER",
        "IDENTITY_ASSERTION_CUSTOM_AS",
        "IDENTITY_ASSERTION_APP_INSTANCE",
    ],
    scopes: list[str],
    *,
    scope_condition: Literal["INCLUDE_ONLY", "EXCLUDE"] = "INCLUDE_ONLY",
    resource_orn: str | None = None,
    resource_indicator: str | None = None,
    auth_server_orn: str | None = None,
):
    print(f"Checking for existing connection for {wlp_id}")

    connections_response = okta_request(
        path=f"/workload-principals/api/v1/ai-agents/{wlp_id}/connections",
        method="GET",
        params={
            "filter": f'connectionType eq "{connection_type}" and resource.orn eq "{resource_orn}"',
        },
    )

    existing_connections = cast(utils.types.ResourceConnectionsResponse, connections_response.json())

    matching_connection = next(
        (connection for connection in existing_connections["data"] if connection["connectionType"] == connection_type and connection.get("resource", {}).get("orn") == resource_orn),
        None,
    )

    if matching_connection:
        print(f"ℹ️ Existing connection found for {wlp_id}, skipping creation!")
        return
    else:
        print(f"ℹ️ No existing connection found for {wlp_id}.")

    print(f"⌛️ Creating {connection_type} connection for {wlp_id}")

    okta_request(
        path=f"/workload-principals/api/v1/ai-agents/{wlp_id}/connections",
        method="POST",
        json={
            "connectionType": connection_type,
            "scopeCondition": scope_condition,
            "scopes": scopes or [],
            "resource": (
                {
                    "orn": resource_orn,
                }
                if (connection_type == "IDENTITY_ASSERTION_A2A_SERVER" and resource_orn is not None)
                else None
            ),
            "authorizationServer": ({"orn": auth_server_orn} if auth_server_orn is not None else None),
            "resourceIndicator": resource_indicator,
        },
    )

    print(f"✅ Connection created successfully!")


# ========== END OF HELPER FUNCTIONS ==========
assert isinstance(client_orn := STATE.get("client_app", {}).get("app", {}).get("orn"), str), "❌ Missing client ORN! Make sure you have completed STEP 3."
assert isinstance(
    auth_server_orn := STATE.get("client_app", {}).get("resource_server", {}).get("orn"),
    str,
), "❌ Missing client auth server ORN! Make sure you have completed STEP 3."
assert isinstance(wlp_id := STATE.get("output").get("PRINCIPAL_A_ID"), str), "❌ Missing PRINCIPAL_A_ID! Make sure you have completed STEP 5."
assert isinstance(resource_orn := STATE.get("agent_j", {}).get("orn"), str), "❌ Missing resource server ORN! Make sure you have completed STEP 5."


create_a2a_resource(wlp_id=wlp_id, auth_server_orn=auth_server_orn)
create_delegation_link(
    client_orn=client_orn,
    resource_orn=resource_orn,
)

print(f"✅ Agent caller successfully added to the workload principal!")

## Step 6: Register Agent K

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

import time

PRINCIPAL_B_NAME = "Agent K"  # @param { type: "string", placeholder: "Enter a name for Principal B." }
# @markdown <sup><code>profile.name</code></sup>
PRINCIPAL_B_DESCRIPTION = "Special Agent K"  # @param { type: "string", placeholder: "Enter a description for Principal B." }
# @markdown <sup><code>profile.description</code></sup>
PRINCIPAL_B_RESOURCE_URI = "https://agent_k"  # @param { type: "string", placeholder: "Enter the resource URI for Principal B." }

agent_k = STATE.get("agent_k")
if agent_k is None:
    agent_k = cast(utils.types.AIAgentState, {})
    STATE["agent_k"] = agent_k

try:
    agent_k = cast(
        utils.types.AIAgentState,
        register_agent(
            agent_k,
            name=PRINCIPAL_B_NAME,
            description=PRINCIPAL_B_DESCRIPTION,
            resource_uri=PRINCIPAL_B_RESOURCE_URI,
        ),
    )
    STATE["agent_k"] = agent_k
finally:
    if agent_k.get("id") != STATE["output"].get("PRINCIPAL_B_ID"):
        STATE["output"]["PRINCIPAL_B_SECRET"] = ""
    STATE["output"]["PRINCIPAL_B_ID"] = agent_k.get("id", "")
    STATE["output"]["PRINCIPAL_B_RESOURCE_URI"] = agent_k.get("resource_uri", "")

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → PRINCIPAL_B_ID: {agent_k.get('id')}")
print(f"    → PRINCIPAL_B_RESOURCE_URI: {agent_k.get('resource_uri')}")

### Configure client registration

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Add%20client%20registration%20details

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown <br><br>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.
# @markdown <br><br>
# @markdown <em>Only <code>client_secret_basic</code> is supported at this time.</em>
# @markdown <br><br>
# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

PRINCIPAL_B_AUTH_METHOD = "client_secret_basic"  # @param ["client_secret_basic"]
# @markdown <sup><code>client.token_endpoint_auth_method</code></sup>

assert isinstance(principal_b_id := STATE["output"].get("PRINCIPAL_B_ID"), str), "PRINCIPAL_B_ID is missing!"

principal_b_secret = create_credentials(principal_b_id)
STATE["output"]["PRINCIPAL_B_SECRET"] = principal_b_secret

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_B_SECRET: {principal_b_secret}")

### Activate Agent K

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Activate%20an%20AI%20agent



In [ ]:
assert isinstance(principal_b_id := STATE["output"].get("PRINCIPAL_B_ID"), str), "PRINCIPAL_B_ID is missing!"

activate_agent(principal_b_id)

## Step : Create a Agent K's Authorization Server

TODO


### 1. Create a custom authorization Server

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-auth-server.

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

DOMAIN_B_AUTH_SERVER_NAME = "Agent K"  # @param { type: "string", placeholder: "Enter a name for your Principal/Domain B\'s authorization server." }
# @markdown <sup><code>name</code></sup>

domain_b_resource_server = create_auth_server(
    name=DOMAIN_B_AUTH_SERVER_NAME,
    description=f"{DOMAIN_B_AUTH_SERVER_NAME} Authorization Server",
    audiences=[STATE.get("output", {}).get("PRINCIPAL_B_RESOURCE_URI", "https://agent_b")],
)

# Generate ORN. We will need it later.
if not domain_b_resource_server.get("orn"):
    domain_b_resource_server.update(
        {
            "orn": get_orn(
                service="idp",
                object_type="authorization_servers",
                object_id=domain_b_resource_server.get("id"),
            )
        }
    )

STATE["agent_k"]["resource_server"] = domain_b_resource_server
STATE["output"]["DOMAIN_B_ISSUER"] = domain_b_resource_server.get("issuer")


print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → DOMAIN_B_ISSUER: {domain_b_resource_server.get('issuer')}")

### 2. Create a custom scope

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-scopes.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

# @markdown <sub><em>Separate multiple scope names with commas.</em></sub>
DOMAIN_B_AUTH_SERVER_SCOPES = "mcp:read"  # @param { type: "string", placeholder: "Enter scope names for your Principal/Domain B\'s required scopes." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
# @markdown <br><sub><em>Separate multiple scope descriptions with commas in the same order as the name(s)</em></sub>
DOMAIN_B_AUTH_SERVER_SCOPE_DESCRIPTIONS = "Read MCP data"  # @param { type: "string", placeholder: "Enter a description for your Principal/Domain B\'s required scopes." }
# @markdown <sup><code>description</code></sup>


if (domain_b_resource_server := STATE["agent_k"].get("resource_server")) is None:
    raise ValueError("❌ Resource server must be created before setting up its scopes. " "Rerun the authorization server setup cell.")

domain_b_scopes = create_scopes(
    auth_server=domain_b_resource_server,
    scopes=DOMAIN_B_AUTH_SERVER_SCOPES,
    descriptions=DOMAIN_B_AUTH_SERVER_SCOPE_DESCRIPTIONS,
    displayNames=DOMAIN_B_AUTH_SERVER_SCOPES,
)
# This is a bit counter-intuitive but go with it. We are storing Domain B's scopes under PRINCIPAL_A_SCOPES because PRINCIPAL A will be using them to call Principal B.
STATE["output"]["PRINCIPAL_A_SCOPES"] = domain_b_scopes

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_A_SCOPES: {STATE['output'].get('PRINCIPAL_A_SCOPES')}")

### 3. Create an access policy & rule

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-access-policies.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

DOMAIN_B_AUTH_SERVER_POLICY_NAME = "Domain B Policy"  # @param { type: "string", placeholder: "Enter a name for your Principal/Domain B\'s access policy." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
DOMAIN_B_AUTH_SERVER_POLICY_DESCRIPTION = "Policy for Client App authorization server"  # @param { type: "string", placeholder: "Enter a description for your Principal/Domain B\'s access policy rule." }
DOMAIN_B_AUTH_SERVER_RULE_NAME = "Invoke Agent"  # @param { type: "string", placeholder: "Enter a name for your Client/Principal/Domain B\'s access policy rule." }
# @markdown <sup><code>description</code></sup>

if (domain_b_resource_server := STATE["agent_k"].get("resource_server")) is None:
    raise ValueError("Resource server not found. Create it first.")

domain_b_resource_server_policy_id = create_policy(
    auth_server=domain_b_resource_server,
    name=DOMAIN_B_AUTH_SERVER_POLICY_NAME,
    description=DOMAIN_B_AUTH_SERVER_POLICY_DESCRIPTION,
    rule_name=DOMAIN_B_AUTH_SERVER_RULE_NAME,
    scopes=[scope.strip() for scope in STATE["output"].get("PRINCIPAL_A_SCOPES", "[*]").strip().strip("[]").split(",") if scope.strip()],
)

domain_b_resource_server["resource_access_policy_id"] = domain_b_resource_server_policy_id

STATE["agent_k"]["resource_server"] = domain_b_resource_server

### Configure Agent Caller

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Configure%20machine%20access

In [ ]:
assert isinstance(
    auth_server_orn := STATE.get("agent_k", {}).get("resource_server", {}).get("orn"),
    str,
), "❌ Missing auth server ORN! Make sure you completed the prior step(s)."
assert isinstance(wlp_id := STATE.get("agent_k", {}).get("id"), str), "❌ Missing Principal B ID! Make sure you have completed STEP 6.1."


create_a2a_resource(wlp_id=wlp_id, auth_server_orn=auth_server_orn)

# Same so far
# SKIP? Check if there is a delegation link between the app and the workload principal
# GET https://atko-ai-admin.oktapreview.com/workload-principals/api/v1/delegation-links?filter=to.resourceOrn+eq+%22orn%3Aoktapreview%3Adirectory%3A00otndie5aBsxxfkL1d7%3Aworkload-principals%3Aai-agents%3Awlp14j1wdb58Pq1kH1d8%22+and+from.clientOrn+eq+%22orn%3Aoktapreview%3Adirectory%3A00otndie5aBsxxfkL1d7%3Aworkload-principals%3Aai-agents%3Awlp14iyj3iplk4whk1d8%22&limit=10

# Check if there is a connection from A1 to A2
# GET https://atko-ai-admin.oktapreview.com/workload-principals/api/v1/ai-agents/wlp14iyj3iplk4whk1d8/connections?filter=connectionType+eq+%22IDENTITY_ASSERTION_A2A_SERVER%22+and+resource.orn+eq+%22orn%3Aoktapreview%3Adirectory%3A00otndie5aBsxxfkL1d7%3Aresource-servers%3Aa2a%3Awlp14j1wdb58Pq1kH1d8%22

# Create delegation link
# POST /delegation-links
# {
#   "from":{
#       "type":"OKTA_AUTHORIZATION_SERVER",
#       "clientOrn":"orn:oktapreview:directory:00otndie5aBsxxfkL1d7:workload-principals:ai-agents:wlp14iyj3iplk4whk1d8",
#       "tokenType":"ACCESS_TOKEN"
#   },
#   "to":{
#       "resourceOrn":"orn:oktapreview:directory:00otndie5aBsxxfkL1d7:workload-principals:ai-agents:wlp14j1wdb58Pq1kH1d8",
#       "authorizationServerOrn":"orn:oktapreview:idp:00otndie5aBsxxfkL1d7:authorization_servers:aus14ihclv4YhSIl61d8"
#   }
# }

# IF NO connection link exists, we need to create one
# POST https://atko-ai-admin.oktapreview.com/workload-principals/api/v1/ai-agents/wlp14iyj3iplk4whk1d8/connections
# {
# "connectionType":"IDENTITY_ASSERTION_A2A_SERVER",
# "resource": {
#   "orn":"orn:oktapreview:directory:00otndie5aBsxxfkL1d7:resource-servers:a2a:wlp14j1wdb58Pq1kH1d8"
# },
# "authorizationServer": {
#   "orn":"orn:oktapreview:idp:00otndie5aBsxxfkL1d7:authorization_servers:aus14ihclv4YhSIl61d8"
# },
# "scopeCondition":"INCLUDE_ONLY",
# "scopes":["mcp:read"]
# }

assert isinstance(agent_j_orn := STATE.get("agent_j", {}).get("orn"), str), "❌ Missing Principal A ORN! Make sure you have completed STEP 5.1."
assert isinstance(agent_k_orn := STATE.get("agent_k", {}).get("orn"), str), "❌ Missing Principal B ORN! Make sure you have completed STEP 6.1."

create_delegation_link(
    client_orn=agent_j_orn,
    resource_orn=agent_k_orn,
    auth_server_orn=auth_server_orn,
)

print(f"✅ Agent caller successfully added to the workload principal!")

### Configure Agent J Connection

TODO

Because Agent K did not exist when we setup Agent J we were unable to add Agent K as a connection. Now that we have created Agent K and configured it to be Agent J's delegate, we need to also add Agent K as a connection [to Agent J]. 

*Adding Agent J as a caller via the Okta Admin Dashboard would normally automatically create a connection for Agent J making this step unnecessary.*

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-secure.htm

In [ ]:
assert isinstance(
    auth_server_orn := STATE.get("agent_k", {}).get("resource_server", {}).get("orn"),
    str,
), "❌ Missing auth server ORN! Make sure you have completed STEP 7."
assert isinstance(wlp_id := STATE.get("agent_j", {}).get("id"), str), "❌ Missing Principal A ID! Make sure you have completed STEP 5.1."
assert isinstance(
    principal_a_scopes := STATE.get("output", {}).get("PRINCIPAL_A_SCOPES"),
    str,
), "❌ Missing Principal A scopes! Make sure you have completed the previous steps."
assert isinstance(principal_b_id := STATE.get("output", {}).get("PRINCIPAL_B_ID"), str), "❌ Missing Principal B ID! Make sure you have completed the previous steps."

create_connection(
    wlp_id=wlp_id,
    connection_type="IDENTITY_ASSERTION_A2A_SERVER",
    scopes=[s.strip() for s in principal_a_scopes.split(",")],
    resource_orn=get_orn(
        service="directory",
        object_type="resource-servers",
        app_name="a2a",
        object_id=principal_b_id,
    ),
    auth_server_orn=auth_server_orn,
)

### Add 'HR Benefits MCP' as a resource connection 

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-secure.htm

In [ ]:
assert isinstance(
    auth_server_orn := STATE.get("resource", {}).get("orn"),
    str,
), "❌ Missing auth server ORN! Make sure you have completed STEP 7."
assert isinstance(wlp_id := STATE.get("agent_j", {}).get("id"), str), "❌ Missing Principal A ID! Make sure you have completed STEP 5.1."
assert isinstance(
    principal_a_scopes := STATE.get("output", {}).get("PRINCIPAL_A_SCOPES"),
    str,
), "❌ Missing Principal A scopes! Make sure you have completed the previous steps."
assert isinstance(principal_b_id := STATE.get("output", {}).get("PRINCIPAL_B_ID"), str), "❌ Missing Principal B ID! Make sure you have completed the previous steps."

create_connection(
    wlp_id=wlp_id,
    connection_type="IDENTITY_ASSERTION_CUSTOM_AS",
    scopes=[s.strip() for s in principal_a_scopes.split(",")],
    resource_orn=get_orn(
        service="directory",
        object_type="resource-servers",
        app_name="a2a",
        object_id=principal_b_id,
    ),
    auth_server_orn=auth_server_orn,
)

## Setup complete!

**Return to your A2A Colab and initialize the configuration to validate your setup.**

In [ ]:
# @title Configuration { display-mode: "form" }
# @markdown _Enter your configuration variables here and click `run` to validate._
# @markdown <br><br> These will be used throughout the notebook.


OKTA_DOMAIN = "https://{your_domain}.oktapreview.com"  # @param {"type":"string", "placeholder": "Enter Okta domain", "description":"Enter your entire Okta domain (including https)"}

CLIENT_ID = ""  # @param {"type":"string","placeholder":"Optional: Enter app client Id"}
APP_ID = ""  # @param {"type":"string", "placeholder": "Enter app ID", "description":"The unique Okta app identifier for the application that will own the registered agent."}

REG_SERVICE_CLIENT_ID = ""  # @param {"type":"string","placeholder":"Enter the service account client Id", "description":"The client ID of the pre-created Okta service app used to call Okta APIs for this exercise."}
REG_SERVICE_PRIVATE_JWK = {}  # @param {"type":"raw", "placeholder": "Enter the JWK as JSON", "description":"The private JWK of the service app used for private_key_jwt authentication to Okta."}

AUTHZ_SERVER_ID = ""  # @param {"type":"string","placeholder":"Enter the authorization server Id.", "description":"The authorization server ID to be used for the agent's managed connection."}

# Agent Configuration
AGENT_NAME = "Demo AI Agent For Registration Demo"  # @param {"type":"string","placeholder":"Enter the name of the agent.", "description":"The name of the agent to be registered.", "default":"Demo AI Agent For Registration Demo"}
AGENT_DESCRIPTION = "A demonstration AI agent for Registration Demo"  # @param {"type":"string","placeholder":"Enter the description of the agent.", "description":"The description of the agent to be registered.", "default":"A demonstration AI agent For Registration Demo"}

import re

# Validate OKTA_DOMAIN format
okta_domain_regex = r"^https:\/\/[a-zA-Z0-9-]+\.(okta|oktapreview|okta-emea)\.com$"
if not re.match(okta_domain_regex, OKTA_DOMAIN):
    raise ValueError(f"Invalid OKTA_DOMAIN: '{OKTA_DOMAIN}'. " "Expected format: 'https://<your-domain>.<okta|oktapreview|okta-emea>.com'")

# Validate other required variables
if not APP_ID:
    raise ValueError("APP_ID cannot be empty.")
if not re.match(r"^0oa[a-zA-Z0-9]+$", APP_ID):
    raise ValueError(f"Invalid APP_ID format: '{APP_ID}'. Expected an Okta app ID starting with '0oa'.")
if not REG_SERVICE_CLIENT_ID:
    raise ValueError("REG_SERVICE_CLIENT_ID cannot be empty.")
if not REG_SERVICE_PRIVATE_JWK:
    raise ValueError("REG_SERVICE_PRIVATE_JWK cannot be empty.")
if not AUTHZ_SERVER_ID:
    raise ValueError("AUTHZ_SERVER_ID cannot be empty.")
if not AGENT_NAME:
    raise ValueError("AGENT_NAME cannot be empty.")
if not AGENT_DESCRIPTION:
    raise ValueError("AGENT_DESCRIPTION cannot be empty.")

print("Configuration loaded!")
print(f"Domain: {OKTA_DOMAIN}")
print(f"App ID: {APP_ID}")
print(f"Authz Server ID: {AUTHZ_SERVER_ID}")

# ==========================
# Helper Functions
# These functions are used throughout the notebook to handle API requests, token generation, and other utility tasks.
import json
import requests
import base64
import time
import uuid
import jwt
from typing import Dict, Any, Optional, Tuple
from urllib.parse import urlparse
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend


def get_okta_partition(okta_domain: str) -> str:
    """Infer ORN partition from Okta domain."""
    hostname = urlparse(okta_domain).hostname or ""
    if hostname.endswith(".oktapreview.com"):
        return "oktapreview"
    if hostname.endswith(".okta-emea.com"):
        return "okta-emea"
    if hostname.endswith(".okta.com"):
        return "okta"
    raise ValueError(f"Unsupported Okta domain for ORN partition derivation: {okta_domain}")


def _generate_jwt_assertion_for_service_token(client_id: str, audience: str, private_jwk: Dict[str, Any]) -> str:
    """
    Generate a JWT assertion for client authentication using private JWK
    Based on RFC 7523 (JSON Web Token (JWT) Profile for OAuth 2.0 Client Authentication)
    """

    if private_jwk is None or not isinstance(private_jwk, dict):
        raise ValueError("private_jwk must be a non-empty dict.")

    required_base_fields = ("n", "e", "d")
    missing_base_fields = [field for field in required_base_fields if not private_jwk.get(field)]
    if missing_base_fields:
        raise ValueError(f"private_jwk is missing required fields: {', '.join(missing_base_fields)}")

    # Get current time
    now = int(time.time())

    # Prepare header
    header = {"kid": private_jwk.get("kid"), "alg": "RS256"}

    # Prepare payload
    payload = {
        "iss": client_id,
        "aud": audience,
        "sub": client_id,
        "exp": now + 60,  # Expires in 60 seconds
    }

    print(f"Generating JWT assertion with payload: {payload} and header: {header}")
    # Convert JWK to RSA private key for signing

    # Helper function to decode base64url to int
    def b64url_to_int(b64url_str: str) -> int:
        """Convert base64url string to integer"""
        # Add padding if needed
        padding = 4 - len(b64url_str) % 4
        if padding != 4:
            b64url_str += "=" * padding
        # Replace URL-safe characters
        b64_str = b64url_str.replace("-", "+").replace("_", "/")
        decoded = base64.b64decode(b64_str)
        return int.from_bytes(decoded, byteorder="big")

    # Extract and decode key components from JWK
    n_int = b64url_to_int(private_jwk["n"])
    e_int = b64url_to_int(private_jwk["e"])
    d_int = b64url_to_int(private_jwk["d"])

    crt_fields = ("p", "q", "dp", "dq", "qi")
    has_all_crt_fields = all(private_jwk.get(field) for field in crt_fields)

    if has_all_crt_fields:
        p_int = b64url_to_int(private_jwk["p"])
        q_int = b64url_to_int(private_jwk["q"])
        dp_int = b64url_to_int(private_jwk["dp"])
        dq_int = b64url_to_int(private_jwk["dq"])
        qi_int = b64url_to_int(private_jwk["qi"])
    else:
        partial_crt_fields = [field for field in crt_fields if private_jwk.get(field)]
        if partial_crt_fields:
            raise ValueError("private_jwk must include either all CRT fields " "('p', 'q', 'dp', 'dq', 'qi') or none of them.")

        p_int, q_int = rsa.rsa_recover_prime_factors(n_int, e_int, d_int)
        dp_int = rsa.rsa_crt_dmp1(d_int, p_int)
        dq_int = rsa.rsa_crt_dmq1(d_int, q_int)
        qi_int = rsa.rsa_crt_iqmp(p_int, q_int)

    private_numbers = rsa.RSAPrivateNumbers(
        p=p_int,
        q=q_int,
        d=d_int,
        dmp1=dp_int,
        dmq1=dq_int,
        iqmp=qi_int,
        public_numbers=rsa.RSAPublicNumbers(e_int, n_int),
    )

    private_key = private_numbers.private_key(backend=default_backend())

    # Encode JWT
    assertion = jwt.encode(payload, private_key, algorithm="RS256", headers=header)

    return assertion


def make_request(
    method: str,
    url: str,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
    timeout_seconds: int = 30,
    endpoint_name: str = "",
) -> requests.Response:
    """Make HTTP request to Okta API with timeout and response diagnostics."""
    req_headers: Dict[str, str] = {"Content-Type": "application/json"}
    if token:
        req_headers["Authorization"] = f"Bearer {token}"
    if headers:
        req_headers.update(headers)

    label = endpoint_name or url
    print(f"[HTTP] {method.upper()} {label}")
    print(req_headers)

    try:
        if method.upper() == "GET":
            response = requests.get(url, headers=req_headers, timeout=timeout_seconds)
        elif method.upper() == "POST":
            response = requests.post(url, json=data, headers=req_headers, timeout=timeout_seconds)
        elif method.upper() == "PUT":
            response = requests.put(url, json=data, headers=req_headers, timeout=timeout_seconds)
        else:
            raise ValueError(f"Unsupported method: {method}")
    except requests.Timeout as timeout_error:
        raise TimeoutError(f"Request timed out after {timeout_seconds}s: {label}") from timeout_error

    print(f"[HTTP] Status: {response.status_code}")
    if response.status_code >= 400:
        print(f"[HTTP] Error body: {response.text}")

    return response


def generate_rsa_key_pair(
    key_size: int = 2048,
) -> Tuple[Dict[str, str], Dict[str, str], str, str]:
    """Generate RSA key pair in JWK and PEM formats."""
    if not isinstance(key_size, int):
        raise TypeError("key_size must be an integer.")
    if key_size < 2048:
        raise ValueError("key_size must be at least 2048 bits.")
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=key_size, backend=default_backend())

    public_key = private_key.public_key()
    public_numbers = public_key.public_numbers()
    private_numbers = private_key.private_numbers()

    def int_to_base64url(value):
        byte_length = (value.bit_length() + 7) // 8
        value_bytes = value.to_bytes(byte_length, byteorder="big")
        return base64.urlsafe_b64encode(value_bytes).rstrip(b"=").decode("utf-8")

    key_id = str(uuid.uuid4())

    jwk_base = {
        "kty": "RSA",
        "kid": key_id,
        "use": "sig",
        "alg": "RS256",
        "n": int_to_base64url(public_numbers.n),
        "e": int_to_base64url(public_numbers.e),
    }

    public_jwk = dict(jwk_base)

    private_jwk = {
        **jwk_base,
        "d": int_to_base64url(private_numbers.d),
        "p": int_to_base64url(private_numbers.p),
        "q": int_to_base64url(private_numbers.q),
        "dp": int_to_base64url(private_numbers.dmp1),
        "dq": int_to_base64url(private_numbers.dmq1),
        "qi": int_to_base64url(private_numbers.iqmp),
    }

    private_pem = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    ).decode("utf-8")

    return public_jwk, private_jwk, private_pem, key_id


print("Helper functions defined!")

In [ ]:
print("Generating RSA Key Pair (2048-bit)...")
PUBLIC_JWK, PRIVATE_JWK, PRIVATE_PEM, KEY_ID = generate_rsa_key_pair(2048)

print("\n[SUCCESS] Key pair generated!")
print(f"\nKey ID: {KEY_ID}")
print(f"\nPublic JWK:")
print(json.dumps(PUBLIC_JWK, indent=2))

### UNDO EVERYTHING

TODO

In [ ]:
1.0

---

## STEP 1: Get Access Token

In [ ]:
print("Step 1: Getting Access Token...\n")

token_data = {
    "grant_type": "client_credentials",
    "client_id": REG_SERVICE_CLIENT_ID,
    "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer",
    "client_assertion": _generate_jwt_assertion_for_service_token(REG_SERVICE_CLIENT_ID, f"{OKTA_DOMAIN}/oauth2/v1/token", REG_SERVICE_PRIVATE_JWK),
    "scope": "okta.aiAgents.manage okta.aiAgents.read okta.governance.resourceOwner.manage okta.apps.manage okta.users.read okta.groups.read okta.authorizationServers.read",
}

resp = requests.post(
    f"{OKTA_DOMAIN}/oauth2/v1/token",
    data=token_data,
    headers={"Content-Type": "application/x-www-form-urlencoded"},
)

body = resp.json()

if resp.status_code == 200:
    access_token = body["access_token"]
    print(f"[✓] Access token obtained - {access_token}")
else:
    print(f"[✗] Failed: {resp.status_code}")
    print(f"{body}")
    raise Exception("Failed to get access token")

---

## STEP 2: Verify App ID

In [ ]:
print("Step 2: Verifying App ID and OIDC app type...\n")

resp = make_request(
    "GET",
    f"{OKTA_DOMAIN}/api/v1/apps/{APP_ID}",
    token=access_token,
    endpoint_name="Verify app by ID",
)

if resp.status_code == 200:
    app = resp.json()
    app_id = app["id"]
    app_name = app.get("label") or app.get("name") or "(unknown)"
    app_type = app.get("name")
    sign_on_mode = app.get("signOnMode")

    is_oidc = app_type == "oidc_client" or sign_on_mode == "OPENID_CONNECT"
    if not is_oidc:
        raise Exception(f"APP_ID '{APP_ID}' is not an OIDC app. " f"Found name='{app_type}', signOnMode='{sign_on_mode}'. " "Use an OIDC app (for example one created as OpenID Connect).")

    print(f"[✓] App ID verified! {app_id}")
    print(f"    App name: {app_name}")
    print(f"    App type: {app_type}, signOnMode: {sign_on_mode}")
else:
    print(resp.text)
    raise Exception(f"Failed to verify app ID: {resp.status_code}")

---

## STEP 3: Create Agent

In [ ]:
print("Step 3: Creating AI Agent...\n")

agent_payload = {
    "appId": app_id,
    "profile": {"name": AGENT_NAME, "description": AGENT_DESCRIPTION},
}

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents",
    token=access_token,
    data=agent_payload,
    endpoint_name="Create AI agent",
)

if resp.status_code == 202:
    print(f"[✓] Agent creation request accepted (HTTP 202)")
    location = resp.headers.get("location")
    if location:
        operation_id = location.split("/")[-1]
        print(f"    Operation ID: {operation_id}")
    else:
        raise Exception("Create agent response did not include an operation location header.")
else:
    print(resp.text)
    raise Exception(f"Failed to create agent: {resp.status_code}")

---

## STEP 4: Get Agent ID from Operation

In [ ]:
print("Step 4: Getting Agent ID from Operation...\n")

max_attempts = 10
sleep_seconds = 2
agent_id = None

for attempt in range(1, max_attempts + 1):
    resp = make_request(
        "GET",
        f"{OKTA_DOMAIN}/workload-principals/api/v1/operations/{operation_id}",
        token=access_token,
        endpoint_name=f"Poll operation status (attempt {attempt}/{max_attempts})",
    )

    if resp.status_code != 200:
        print(resp.text)
        raise Exception(f"Failed to fetch operation status: {resp.status_code}")

    operation = resp.json()
    status = operation.get("status")
    print(f"    Attempt {attempt}: operation status = {status}")

    if status == "COMPLETED":
        agent_id = operation["resource"]["id"]
        print(f"[✓] Agent ID retrieved! {agent_id}")
        break

    if status in {"FAILED", "CANCELED"}:
        raise Exception(f"Operation finished with terminal status '{status}': {json.dumps(operation, indent=2)}")

    time.sleep(sleep_seconds)

if not agent_id:
    raise TimeoutError(f"Operation did not complete after {max_attempts} attempts.")

---

## STEP 5: Add Public Key to Agent

In [ ]:
print("Step 5: Adding Public Key to Agent...\n")

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents/{agent_id}/credentials/jwks",
    token=access_token,
    data=PUBLIC_JWK,
)

if resp.status_code == 201:
    key_response = resp.json()
    print(f"[✓] Public key added! Kid: {key_response['kid']}")
else:
    raise Exception(f"Failed: {resp.status_code}")

---

## STEP 6: Get Org ID

In [ ]:
print("Step 6: Getting Org ID...\n")

resp = make_request(
    "GET",
    f"{OKTA_DOMAIN}/.well-known/okta-organization",
    headers={"Accept": "application/json"},
)

if resp.status_code == 200:
    org_data = resp.json()
    org_id = org_data["id"]
    print(f"[✓] Org ID: {org_id}")
else:
    raise Exception(f"Failed: {resp.status_code}")

---

## STEP 7: Get Owner Group ID

In [ ]:
print("Step 7: Getting Owner Group ID...\n")

OWNER_GROUP_NAME = "Beta Admins"
resp = make_request(
    "GET",
    f"{OKTA_DOMAIN}/api/v1/groups?q={OWNER_GROUP_NAME}",
    token=access_token,
    endpoint_name="List groups by query",
)

owner_group_id = None
if resp.status_code == 200:
    groups = resp.json()
    exact_group_matches = [g for g in groups if g.get("profile", {}).get("name") == OWNER_GROUP_NAME]

    if len(exact_group_matches) == 1:
        owner_group_id = exact_group_matches[0]["id"]
        print(f"[✓] Owner group found! ID: {owner_group_id}")
    elif len(exact_group_matches) > 1:
        print("[!] Multiple exact owner group matches found:")
        for group in exact_group_matches:
            print(f"    - {group.get('id')}: {group.get('profile', {}).get('name')}")
        raise Exception("Multiple exact owner group matches found. Use a more specific group query.")
    else:
        raise Exception("No exact owner group match found for '{OWNER_GROUP_NAME}'.")
else:
    print(resp.text)
    raise Exception(f"Could not fetch groups: {resp.status_code}")

---

## STEP 8: Assign Owner to Agent

In [ ]:
print("Step 8: Assigning Owner to Agent...\n")

if owner_group_id:
    owner_payload = {
        "resourceOrns": [f"orn:okta:directory:{org_id}:workload-principals:ai-agents:{agent_id}"],
        "principalOrns": [f"orn:okta:directory:{org_id}:groups:{owner_group_id}"],
    }

    print(owner_payload)

    resp = make_request(
        "POST",
        f"{OKTA_DOMAIN}/governance/api/v1/resource-owners",
        token=access_token,
        data=owner_payload,
        endpoint_name="Assign owner to AI agent",
    )

    if resp.status_code in [200, 204]:
        print(f"[✓] Owner assigned!")
    else:
        print(resp.text)
        raise Exception(f"Owner assignment failed with status: {resp.status_code}")
else:
    raise Exception("Owner assignment failed because no owner group was resolved.")

---

## STEP 9: Activate Agent

In [ ]:
print("Step 9: Activating Agent...\n")

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents/{agent_id}/lifecycle/activate",
    token=access_token,
    data={},
)

if resp.status_code == 202:
    print(f"[✓] Agent activated!")
else:
    raise Exception(f"Failed: {resp.status_code}")

---

## STEP 10: Add Managed Connection

In [ ]:
print("Step 10: Adding Managed Connection...\n")

okta_partition = get_okta_partition(OKTA_DOMAIN)
print(f"Using ORN partition derived from domain: {okta_partition}")

connection_payload = {
    "connectionType": "IDENTITY_ASSERTION_CUSTOM_AS",
    "authorizationServer": {"orn": f"orn:{okta_partition}:idp:{org_id}:authorization_servers:{AUTHZ_SERVER_ID}"},
    "scopeCondition": "INCLUDE_ONLY",
    "scopes": ["read:data"],
}

print(f"Payload: {json.dumps(connection_payload, indent=2)}\n")

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents/{agent_id}/connections",
    token=access_token,
    data=connection_payload,
    endpoint_name="Create managed connection",
)

if resp.status_code in [200, 201]:
    connection_response = resp.json()
    print(f"[✓] Managed connection created!")
    print(f"    Connection ID: {connection_response['id']}")
    print(f"    Connection Type: {connection_response['connectionType']}")
    print(f"    Scopes: {', '.join(connection_response.get('scopes', []))}")
else:
    print(f"[✗] Failed: {resp.status_code}")
    print(resp.text)
    raise Exception("Failed to create managed connection")

---

## Summary

In [ ]:
print("\n" + "=" * 80)
print("AI AGENT REGISTRATION COMPLETE")
print("=" * 80)
print(f"\n✓ Agent ID: {agent_id}")
print(f"✓ Agent Name: {AGENT_NAME}")
print(f"✓ Key ID: {KEY_ID}")
print(f"✓ Connection Type: IDENTITY_ASSERTION_CUSTOM_AS")
print(f"\n" + "=" * 80)